# Projekt mastermind loesung

<div style="display:flex;justify-content:space-between;align-items:center;background:#0d0d0f;border:1px solid #1e1e24;border-left:4px solid #4fc3f7;border-radius:6px;padding:clamp(1rem,2.5vw,1.8rem) clamp(1.2rem,3vw,2.4rem);margin-bottom:2rem;position:relative;overflow:hidden;box-shadow:0 4px 32px rgba(0,0,0,0.5);font-family:'Segoe UI',sans-serif;">
<div style="position:absolute;top:0;left:0;right:0;bottom:0;background:radial-gradient(ellipse at 0% 50%,rgba(79,195,247,0.07) 0%,transparent 60%);pointer-events:none;"></div>
<div style="display:flex;flex-direction:column;gap:0.3rem;">
<p style="font-size:clamp(1.3rem,3.5vw,2.4rem);color:#f0f0f5;margin:0;line-height:1.1;font-weight:700;letter-spacing:-0.01em;">Mastermind: Von der Idee zur Implementierung</p>
<p style="font-size:clamp(0.75rem,1.6vw,1rem);color:#7a7a90;margin:0;letter-spacing:0.04em;font-weight:300;">Development Expert Python &nbsp;|&nbsp; Kapitel 13: Abschlussprojekt &nbsp;|&nbsp; Loesung und Herleitung</p>
</div>
</div>

> **[Kursinhalt]** Dieses Notebook zeigt den vollstaendigen Loesungsweg -- von der Spielidee bis zur lauffaehigen Implementierung. Es ersetzt nicht das eigenstaendige Loesen, sondern dient der Nachbereitung und dem Verstaendnis der Entwurfsentscheidungen.

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
1. Das Spiel verstehen -- bevor eine Zeile Code entsteht
</span>
</div>

Mastermind ist ein Zwei-Spieler-Ratespiel aus dem Jahr 1970. Ein Spieler legt einen geheimen Farbcode fest -- vier Farben aus sechs moeglichen, Wiederholungen erlaubt. Der andere Spieler versucht diesen Code in maximal zehn Versuchen zu erraten.

Nach jedem Versuch erhaelt der Ratende eine Rueckmeldung:

- Ein **schwarzer Pin** fuer jede Farbe an der **richtigen Position**
- Ein **weisser Pin** fuer jede Farbe die im Code vorkommt aber an der **falschen Position** steht

Das Ziel: vier schwarze Pins -- der Code ist vollstaendig erraten.

**Bevor wir programmieren, spielen wir ein Beispiel manuell durch:**

```
Geheimcode:  rot   blau  rot   gruen

Versuch 1:   rot   rot   blau  gelb     ->  S=1, W=1
Versuch 2:   blau  blau  rot   rot      ->  S=1, W=2
Versuch 3:   rot   blau  rot   gruen    ->  S=4, W=0  (gewonnen!)
```

Was faellt auf?
- Versuch 1: `rot` an Position 0 trifft (S=1). Das `rot` an Position 1 des Versuchs trifft nicht exakt -- aber es gibt noch ein `rot` im Code (Position 2). Also W=1.
- Die Bewertung sagt nie *welche* Position stimmt, nur *wie viele*.

Daraus folgt schon die erste Programmierfrage: **Wie berechnen wir S und W korrekt, wenn eine Farbe mehrfach vorkommt?**

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
2. Das Kernproblem isolieren
</span>
</div>

Bevor man ein Projekt baut, lohnt es sich zu fragen: **Was ist die schwierigste Teilaufgabe?** Alles andere ist Verpackung.

Bei Mastermind ist das die Bewertungslogik. Den Code erzeugen ist trivial (`random.choices`). Eingaben lesen ist Fleissarbeit. Aber die korrekte Bewertung mit Wiederholungen -- das ist der einzige Punkt wo man echte Fehler bauen kann.

**Warum ist es schwierig?**

Betrachte diesen Fall:

```
Code:    rot  rot  blau  gruen
Versuch: rot  rot  rot   rot
```

Intuitiv moechte man sagen: der Code hat 2x `rot`, der Versuch hat 4x `rot` -- also 2 schwarze Treffer und... wie viele weisse? Die Antwort muss 0 sein. Die beiden `rot` im Versuch an Positionen 2 und 3 finden kein entsprechendes `rot` mehr im Code -- beide wurden bereits als schwarz verbraucht.

Oder dieser:

```
Code:    rot  blau  rot   gruen
Versuch: rot  rot   blau  gelb
```

Erwartetes Ergebnis: S=1, W=2. Warum 2?
- Position 0: `rot == rot` -> schwarz, beide verbraucht
- Position 1 (Versuch): `rot` -- kein exakter Treffer, aber Code hat noch `rot` an Position 2 -> weiss
- Position 2 (Versuch): `blau` -- kein exakter Treffer, aber Code hat `blau` an Position 1 -> weiss
- Position 3 (Versuch): `gelb` -- nicht im Code

Der Schluessel: **schwarze Treffer muessen zuerst gezaehlt und verbraucht werden**, bevor weisse gesucht werden. Sonst zaehlt ein exakter Treffer doppelt.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
3. Erster Ansatz: naiv und falsch
</span>
</div>

Der naheliegendste erste Versuch: Position fuer Position durchgehen und zaehlen.

In [ ]:
# --- Versuch 1: der naive Ansatz ---
# Wir gehen Position fuer Position durch.
# Schwarz: gleiche Position, gleiche Farbe.
# Weiss: Farbe kommt irgendwo im Code vor.

def bewerte_v1(code, versuch):
    schwarz = 0
    weiss   = 0
    for i in range(len(code)):
        if versuch[i] == code[i]:
            schwarz += 1
        elif versuch[i] in code:   # Fehler steckt hier
            weiss += 1
    return schwarz, weiss


# Dieser einfache Fall funktioniert noch:
print(bewerte_v1(['r','b','g','y'], ['r','x','g','x']))  # erwartet: (2, 0)

# Hier versagt der naive Ansatz:
# Code hat 1x 'r'. Versuch hat 4x 'r'. Erwartetes Ergebnis: (1, 0)
print(bewerte_v1(['r','b','g','y'], ['r','r','r','r']))  # erwartet: (1, 0) -- liefert aber?

# Und dieser:
# Code: r b r g | Versuch: r r b y
# Erwartet: (1, 2) -- 'r' an Pos 0 schwarz, 'r' an Pos 1 weiss (Code hat noch r an Pos 2), 'b' weiss
print(bewerte_v1(['r','b','r','g'], ['r','r','b','y']))  # erwartet: (1, 2) -- liefert aber?

Der naive Ansatz hat zwei Fehler:

1. `versuch[i] in code` prueft ob die Farbe *irgendwo* im original-Code vorkommt -- egal ob diese Position schon als schwarz gezaehlt wurde. Eine schwarz gezaehlte Farbe kann nochmal als weiss auftauchen.

2. Es wird nicht gezaehlt wie oft eine Farbe noch verfuegbar ist. Steht `r` einmal im Code aber viermal im Versuch, wuerde der naive Ansatz bis zu dreimal weiss vergeben.

**Die Erkenntnis:** Wir brauchen ein Konzept von *verbrauchten* Positionen.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
4. Zweiter Ansatz: mit verbrauchten Positionen
</span>
</div>

Die Idee: wir arbeiten nicht mit den Originallisten, sondern mit Kopien. Jede Position die gezaehlt wurde -- egal ob schwarz oder weiss -- markieren wir als `None`. So kann sie nicht nochmal zaehlen.

Ausserdem trennen wir die zwei Phasen sauber:
1. Erst alle schwarzen Treffer einsammeln (exakte Matches)
2. Dann erst nach weissen Treffern suchen -- aber nur in den noch nicht verbrauchten Positionen

In [ ]:
# --- Versuch 2: Zweiphasen-Algorithmus mit Verbrauchsmarkierung ---

def bewerte_v2(code, versuch):
    # Kopien anlegen -- die Originale werden nicht veraendert
    code_rest    = list(code)
    versuch_rest = list(versuch)

    # Phase 1: Schwarze Treffer
    # Exakte Uebereinstimmungen finden und verbrauchen
    schwarz = 0
    for i in range(len(code_rest)):
        if versuch_rest[i] == code_rest[i]:
            schwarz += 1
            code_rest[i]    = None   # verbraucht
            versuch_rest[i] = None   # verbraucht

    # Phase 2: Weisse Treffer
    # Jede noch nicht verbrauchte Versuchsposition: kommt die Farbe noch im Code vor?
    weiss = 0
    for i in range(len(versuch_rest)):
        farbe = versuch_rest[i]
        if farbe is None:        # bereits als schwarz verbraucht
            continue
        if farbe in code_rest:   # Farbe noch im restlichen Code?
            weiss += 1
            code_rest[code_rest.index(farbe)] = None   # diese Fundstelle verbrauchen

    return schwarz, weiss


# Dieselben Testfaelle nochmal:
print(bewerte_v2(['r','b','g','y'], ['r','r','r','r']))   # erwartet: (1, 0)
print(bewerte_v2(['r','b','r','g'], ['r','r','b','y']))   # erwartet: (1, 2)
print(bewerte_v2(['r','b','g','y'], ['b','g','y','r']))   # erwartet: (0, 4)
print(bewerte_v2(['r','r','b','g'], ['r','r','r','r']))   # erwartet: (2, 0)

Alle Testfaelle stimmen. Warum funktioniert das?

Der entscheidende Schritt ist `code_rest[code_rest.index(farbe)] = None` in Phase 2. Wenn `r` einmal im Code steht und zweimal im Versuch vorkommt (beide nicht-schwarz), findet der erste Weiss-Treffer das `r` und verbraucht es. Der zweite Versuch findet kein `r` mehr -- kein doppelter Weiss-Treffer.

**Diese Funktion ist der Kern des gesamten Projekts.** Alles andere -- Code erzeugen, Eingabe lesen, GUI zeichnen -- ist Infrastruktur um diese eine Funktion herum.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
5. Den Algorithmus Schritt fuer Schritt verfolgen
</span>
</div>

In [ ]:
# Wir instrumentieren bewerte_v2 mit print()-Ausgaben um den Ablauf zu verfolgen.
# Das ist eine klassische Debugging-Technik: print-Tracing.

def bewerte_trace(code, versuch):
    print(f"  Code:    {code}")
    print(f"  Versuch: {versuch}")
    print()

    code_rest    = list(code)
    versuch_rest = list(versuch)

    schwarz = 0
    print("  --- Phase 1: Schwarze Treffer ---")
    for i in range(len(code_rest)):
        if versuch_rest[i] == code_rest[i]:
            print(f"  Position {i}: '{versuch_rest[i]}' == '{code_rest[i]}' -> SCHWARZ")
            schwarz += 1
            code_rest[i]    = None
            versuch_rest[i] = None
        else:
            print(f"  Position {i}: '{versuch_rest[i]}' != '{code_rest[i]}'")

    print(f"  code_rest nach Phase 1:    {code_rest}")
    print(f"  versuch_rest nach Phase 1: {versuch_rest}")
    print()

    weiss = 0
    print("  --- Phase 2: Weisse Treffer ---")
    for i in range(len(versuch_rest)):
        farbe = versuch_rest[i]
        if farbe is None:
            print(f"  Position {i}: bereits verbraucht")
            continue
        if farbe in code_rest:
            print(f"  Position {i}: '{farbe}' in code_rest -> WEISS")
            weiss += 1
            code_rest[code_rest.index(farbe)] = None
        else:
            print(f"  Position {i}: '{farbe}' nicht im restlichen Code")

    print()
    print(f"  => Ergebnis: Schwarz={schwarz}, Weiss={weiss}")
    print()
    return schwarz, weiss


# Der schwierige Fall -- jetzt sieht man genau warum es (1, 2) ergibt:
bewerte_trace(['r', 'b', 'r', 'g'], ['r', 'r', 'b', 'y'])

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
6. Den Geheimcode erzeugen
</span>
</div>

Der Geheimcode ist eine zufaellige Auswahl von 4 Farben aus 6 -- mit Wiederholung erlaubt. `random.choices` loest das in einer Zeile.

In [ ]:
import random

FARBEN = ['rot', 'blau', 'gruen', 'gelb', 'orange', 'lila']

# random.choices zieht k Elemente MIT Zuruecklegen (Wiederholungen moeglich)
# random.sample waere OHNE Zuruecklegen -- das waere hier falsch
code = random.choices(FARBEN, k=4)
print(f"Geheimcode: {code}")

# Mit seed: reproduzierbar fuer Tests
random.seed(42)
print(f"Mit seed=42: {random.choices(FARBEN, k=4)}")
random.seed(42)
print(f"Mit seed=42: {random.choices(FARBEN, k=4)}"  # identisch)

Der `seed`-Parameter ist fuer Tests unverzichtbar: ohne ihn ist der Code bei jedem Aufruf anders -- Tests werden unzuverlaessig. Mit `seed=42` bekommt man immer denselben Code, egal wie oft man ihn aufruft.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
7. Eingabe lesen und validieren
</span>
</div>

Der Spieler gibt eine Eingabe wie `rot blau rot gruen` ein. Diese muss in eine Liste umgewandelt und validiert werden. Fehlerhafte Eingaben sollen eine verstaendliche Meldung liefern -- keinen Absturz.

Die Funktion wirft `ValueError` bei ungueltiger Eingabe. Der Aufrufer (Konsole oder GUI) faengt den Fehler ab und zeigt die Meldung an.

In [ ]:
ABKUERZUNGEN = {'r': 'rot', 'b': 'blau', 'g': 'gruen',
                'ge': 'gelb', 'o': 'orange', 'l': 'lila'}

def normalisiere_eingabe(text, laenge=4):
    """
    Parst einen Eingabe-String.
    Wirft ValueError bei ungueltiger Eingabe.
    Gibt eine bereinigte Farbliste zurueck.
    """
    # strip() entfernt fuehrendes/nachfolgendes Whitespace
    # lower() macht gross/klein egal
    # split() teilt an Leerzeichen auf
    teile = text.strip().lower().split()

    if len(teile) != laenge:
        raise ValueError(
            f"Bitte genau {laenge} Farben eingeben. Du hast {len(teile)} eingegeben."
        )

    ergebnis = []
    for token in teile:
        if token in ABKUERZUNGEN:
            ergebnis.append(ABKUERZUNGEN[token])
        elif token in FARBEN:
            ergebnis.append(token)
        else:
            raise ValueError(f"'{token}' ist keine gueltige Farbe.")

    return ergebnis


# Valide Eingaben:
print(normalisiere_eingabe('rot blau gruen gelb'))
print(normalisiere_eingabe('R B G Ge'))           # Grossschreibung
print(normalisiere_eingabe('r b g ge'))           # Abkuerzungen

# Ungueltige Eingaben -- ValueError wird gefangen und ausgegeben:
for test in ['rot blau', 'rot blau gruen pink', 'rot blau gruen gelb orange']:
    try:
        normalisiere_eingabe(test)
    except ValueError as e:
        print(f"Fehler: {e}")

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
8. Entwurfsentscheidung: Warum eine Spiel-Klasse?
</span>
</div>

Wir haben jetzt alle Bausteine als Funktionen. Man koennte das Spiel direkt mit Variablen in einer Schleife bauen:

```python
code     = erzeuge_code()
versuche = []
gewonnen = False

while len(versuche) < 10 and not gewonnen:
    eingabe = input('...')
    ...
```

Das funktioniert fuer eine Konsolenversion. Aber sobald eine GUI dazukommt, wird es problematisch: Die GUI kann nicht einfach `input()` aufrufen -- sie wartet auf Klicks. Der Spielzustand muss irgendwo gespeichert sein, und zwar unabhaengig davon ob gerade Konsole oder GUI laeuft.

Die Loesung: eine Klasse `Spiel` die den **Zustand** kapselt. Sie weiss:
- Was der Geheimcode ist
- Wie viele Versuche gemacht wurden
- Ob das Spiel gewonnen oder verloren ist

Sowohl Konsole als auch GUI benutzen dieselbe Klasse -- sie rufen nur `spiel.versuch_ausfuehren(...)` auf und lesen den Zustand aus. **Die Klasse weiss nichts von der Darstellung.**

In [ ]:
# Die Spiel-Klasse -- Zustandscontainer fuer eine laufende Partie

class Spiel:
    def __init__(self, laenge=4, max_versuche=10, seed=None):
        self.laenge       = laenge
        self.max_versuche = max_versuche
        # Code beim Start erzeugen -- danach unveraenderlich
        random.seed(seed)
        self.geheimcode   = random.choices(FARBEN, k=laenge)
        self.versuche     = []   # alle bisherigen Versuche
        self.bewertungen  = []   # Bewertung zu jedem Versuch
        self.gewonnen     = False

    @property
    def anzahl_versuche(self):
        return len(self.versuche)

    @property
    def ist_beendet(self):
        return self.gewonnen or self.anzahl_versuche >= self.max_versuche

    def versuch_ausfuehren(self, versuch):
        """Fuehrt einen Versuch aus. Gibt (schwarz, weiss) zurueck."""
        if self.ist_beendet:
            raise RuntimeError('Das Spiel ist bereits beendet.')

        schwarz, weiss = bewerte_v2(self.geheimcode, versuch)
        self.versuche.append(versuch)
        self.bewertungen.append((schwarz, weiss))

        if schwarz == self.laenge:
            self.gewonnen = True

        return schwarz, weiss


# Demonstration: ein komplettes Spiel mit bekanntem Code
spiel = Spiel(seed=42)
print(f"Code (normalerweise geheim): {spiel.geheimcode}")
print()

versuche = [
    ['rot',    'rot',    'rot',    'rot'],
    ['blau',   'blau',   'blau',   'blau'],
    ['lila',   'orange', 'lila',   'blau'],   # nah dran...
]

for v in versuche:
    s, w = spiel.versuch_ausfuehren(v)
    print(f"Versuch {spiel.anzahl_versuche}: {v}  ->  S={s}, W={w}")

print(f"\nSpiel beendet: {spiel.ist_beendet}, Gewonnen: {spiel.gewonnen}")

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
9. Modularer Aufbau: Logik von Darstellung trennen
</span>
</div>

Das fertige Projekt besteht aus drei Dateien. Die wichtigste Entwurfsentscheidung: **`logik.py` hat kein `print()`, kein `input()` und kein `tkinter`.** Sie gibt Werte zurueck und erwartet Werte -- die Darstellung entscheidet was damit passiert.

```
logik.py                    konsole.py              gui.py
                                                    
erzeuge_code()  <---------  spiel = Spiel()  <---  app = MastermindApp()
bewerte()       <---------  versuch_ausfuehren()    versuch_absenden()
Spiel           <---------  print(verlauf)          treeview.update()
                                                    
Kein I/O.                   Text-I/O.              GUI-I/O.
Keine GUI.                  Kein tkinter.           Kein print().
```

Diese Trennung hat einen konkreten Vorteil: `logik.py` kann ohne laufendes Fenster getestet werden. Die Tests in `test_logik.py` importieren nur `logik.py` -- kein GUI, kein Eingabeprompt.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
10. Die fertige logik.py -- vollstaendig und kommentiert
</span>
</div>

In [ ]:
# logik.py -- der vollstaendige, produktionsreife Code
# Dieser Block entspricht exakt der Datei die im Projekt verwendet wird.

import random

FARBEN             = ['rot', 'blau', 'gruen', 'gelb', 'orange', 'lila']
ABKUERZUNGEN       = {'r':'rot','b':'blau','g':'gruen','ge':'gelb','o':'orange','l':'lila'}
STANDARD_LAENGE    = 4
STANDARD_VERSUCHE  = 10


def erzeuge_code(laenge=STANDARD_LAENGE, farben=None, seed=None):
    if farben is None:
        farben = FARBEN
    if seed is not None:
        random.seed(seed)
    return random.choices(farben, k=laenge)


def normalisiere_eingabe(text, laenge=STANDARD_LAENGE, farben=None):
    if farben is None:
        farben = FARBEN
    teile = text.strip().lower().split()
    if len(teile) != laenge:
        raise ValueError(
            f"Bitte genau {laenge} Farben eingeben. Du hast {len(teile)} eingegeben."
        )
    ergebnis = []
    for token in teile:
        if token in ABKUERZUNGEN:
            ergebnis.append(ABKUERZUNGEN[token])
        elif token in farben:
            ergebnis.append(token)
        else:
            raise ValueError(f"'{token}' ist keine gueltige Farbe.")
    return ergebnis


def bewerte(geheimcode, versuch):
    assert len(geheimcode) == len(versuch)
    code_rest    = list(geheimcode)
    versuch_rest = list(versuch)

    schwarz = 0
    for i in range(len(code_rest)):
        if versuch_rest[i] == code_rest[i]:
            schwarz += 1
            code_rest[i] = versuch_rest[i] = None

    weiss = 0
    for i in range(len(versuch_rest)):
        farbe = versuch_rest[i]
        if farbe is None:
            continue
        if farbe in code_rest:
            weiss += 1
            code_rest[code_rest.index(farbe)] = None

    return schwarz, weiss


class Spiel:
    def __init__(self, laenge=STANDARD_LAENGE, max_versuche=STANDARD_VERSUCHE,
                 farben=None, seed=None):
        self.laenge       = laenge
        self.max_versuche = max_versuche
        self.farben       = farben or FARBEN
        self.geheimcode   = erzeuge_code(laenge=laenge, farben=self.farben, seed=seed)
        self.versuche     = []
        self.bewertungen  = []
        self.gewonnen     = False

    @property
    def anzahl_versuche(self): return len(self.versuche)

    @property
    def versuche_uebrig(self): return self.max_versuche - self.anzahl_versuche

    @property
    def ist_beendet(self): return self.gewonnen or self.anzahl_versuche >= self.max_versuche

    def versuch_ausfuehren(self, versuch):
        if self.ist_beendet:
            raise RuntimeError('Das Spiel ist bereits beendet.')
        ergebnis = bewerte(self.geheimcode, versuch)
        schwarz, _ = ergebnis
        self.versuche.append(versuch)
        self.bewertungen.append(ergebnis)
        if schwarz == self.laenge:
            self.gewonnen = True
        return ergebnis

    def verlauf(self):
        return [
            {'nr': i+1, 'versuch': v, 'schwarz': s, 'weiss': w}
            for i, (v, (s, w)) in enumerate(zip(self.versuche, self.bewertungen))
        ]


print('logik.py geladen.')
print(f'Testcode: {erzeuge_code(seed=0)}')
print(f'Bewertung: {bewerte(["rot","blau","gruen","gelb"],["rot","blau","gruen","gelb"]))}')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
11. Die Konsolenversion -- Logik in eine Spielschleife einbetten
</span>
</div>

Die Konsolenversion verbindet `Spiel` mit `input()` und `print()`. Der einzige Trick: die Eingabe sitzt in einer inneren Schleife die erst verlassen wird wenn eine gueltige Eingabe kommt.

In [ ]:
# Vereinfachte Konsolenversion -- illustriert die Grundstruktur.
# Die vollstaendige Version liegt in konsole.py.

def spielrunde_mini():
    spiel = Spiel()
    print(f"Mastermind -- errate den Code ({spiel.laenge} Farben, {spiel.max_versuche} Versuche)")
    print(f"Farben: {', '.join(FARBEN)}")
    print()

    while not spiel.ist_beendet:
        # Innere Schleife: laeuft bis eine gueltige Eingabe kommt
        while True:
            try:
                text    = input(f"Versuch {spiel.anzahl_versuche + 1}: ")
                versuch = normalisiere_eingabe(text)
                break   # gueltige Eingabe -> innere Schleife verlassen
            except ValueError as e:
                print(f"  Fehler: {e}")

        schwarz, weiss = spiel.versuch_ausfuehren(versuch)
        print(f"  -> S={schwarz}, W={weiss}")

    if spiel.gewonnen:
        print(f"\nGewonnen in {spiel.anzahl_versuche} Versuch(en)!")
    else:
        print(f"\nLeider verloren. Code war: {spiel.geheimcode}")


# spielrunde_mini()   # auskommentiert -- wuerde input() aufrufen

Das Muster der inneren `while True`-Schleife ist Standard fuer validierte Eingaben in Python:

```python
while True:
    try:
        wert = verarbeite(input(...))
        break          # erfolgreich -> Schleife verlassen
    except ValueError:
        print('...')   # Fehler anzeigen, nochmal fragen
```

Die aeussere Schleife (`while not spiel.ist_beendet`) ist die eigentliche Spielschleife -- sie laeuft bis das Spiel vorbei ist.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
12. Die GUI-Version -- dieselbe Logik, andere Darstellung
</span>
</div>

In der GUI gibt es kein `input()` und keine `while`-Schleife. Stattdessen reagiert die Anwendung auf **Ereignisse**: ein Klick auf einen Farbkreis, ein Klick auf "Absenden".

Die Struktur ist dieselbe -- nur umgekehrt:

```
Konsole:                         GUI:

while not spiel.ist_beendet:     def _farbe_gewaehlt(farbe):
    versuch = input()                aktuelle_eingabe.append(farbe)
    bewerte(versuch)                 zeichne_eingabe()
    print(ergebnis)              
                                 def _versuch_absenden():
                                     spiel.versuch_ausfuehren(eingabe)
                                     zeichne_zeile(ergebnis)
```

In beiden Faellen ist der zentrale Aufruf `spiel.versuch_ausfuehren(versuch)` -- die Logik ist identisch. Nur die Eingabe und die Ausgabe unterscheiden sich.

**Das Canvas-Prinzip in der GUI:**

Das Spielfeld wird auf einem `tk.Canvas` gezeichnet. Jeder Kreis ist ein Oval-Element mit einer ID. Um eine Farbe zu aendern, wird `canvas.itemconfig(id, fill=farbe)` aufgerufen -- das Oval bleibt bestehen, nur seine Farbe aendert sich.

```python
# Beim Aufbau: Oval erstellen, ID merken
oid = canvas.create_oval(x0, y0, x1, y1, fill='#2a2a3a')
kreis_ids.append(oid)

# Wenn der Spieler eine Farbe waehlt: Oval einfarben
canvas.itemconfig(kreis_ids[position], fill='#e53935')  # rot
```

Die `VersuchsZeile`-Klasse in `gui.py` kapselt genau das: sie zeichnet sich initial mit leeren Kreisen und bietet `setze_farben()` und `setze_bewertung()` an. Die `MastermindApp` ruft diese Methoden auf -- sie muss nicht wissen wie die Kreise intern gespeichert sind.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
13. Tests schreiben -- warum und wie
</span>
</div>

Tests sind kein Luxus -- sie sind das Werkzeug mit dem man sicher stellen kann dass `bewerte()` korrekt funktioniert, auch fuer die unintuitiven Faelle mit Wiederholungen.

Der wichtigste Testfall ist genau der den wir in Abschnitt 2 manuell durchgerechnet haben:

In [ ]:
# Die wichtigsten Testfaelle fuer bewerte() -- alle manuell nachvollziehbar

testfaelle = [
    # (code, versuch, erwartetes_ergebnis, beschreibung)
    (
        ['r','b','g','y'], ['r','b','g','y'],
        (4, 0),
        'Perfekter Treffer'
    ),
    (
        ['r','b','g','y'], ['b','g','y','r'],
        (0, 4),
        'Alle Farben enthalten, alle verschoben'
    ),
    (
        ['r','r','b','g'], ['r','r','r','r'],
        (2, 0),
        'Code hat 2x r -- Versuch hat 4x r -- nur 2 zaehlen'
    ),
    (
        ['r','b','r','g'], ['r','r','b','y'],
        (1, 2),
        'Der schwierige Fall aus Abschnitt 2'
    ),
    (
        ['r','b','g','y'], ['r','b','g','y'],
        (4, 0),
        'Schwarz darf nicht nochmal als Weiss zaehlen'
    ),
]

alle_ok = True
for code, versuch, erwartet, beschreibung in testfaelle:
    ergebnis = bewerte(code, versuch)
    status   = 'OK' if ergebnis == erwartet else 'FEHLER'
    if status == 'FEHLER':
        alle_ok = False
    print(f"[{status}] {beschreibung}")
    if status == 'FEHLER':
        print(f"       erwartet {erwartet}, bekommen {ergebnis}")

print()
print('Alle Tests bestanden.' if alle_ok else 'Es gibt Fehler!')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
14. Rueckblick: der Weg vom Problem zum Programm
</span>
</div>

Dieser Weg hat sich durch das gesamte Notebook gezogen:

| Schritt | Was wir getan haben |
|---|---|
| **1** | Das Spiel manuell gespielt und verstanden was berechnet werden muss |
| **2** | Das Kernproblem isoliert: Bewertung mit Wiederholungen |
| **3** | Einen naiven Ansatz gebaut -- und seine Fehler gefunden |
| **4** | Den korrekten Zweiphasen-Algorithmus entwickelt |
| **5** | Den Algorithmus mit print-Tracing Schritt fuer Schritt verfolgt |
| **6** | Die einfachen Teile (Code erzeugen, Eingabe) gebaut |
| **7** | Die Architekturentscheidung getroffen: `Spiel`-Klasse als Zustandscontainer |
| **8** | Logik, Konsole und GUI sauber getrennt |
| **9** | Tests fuer die kritischste Funktion geschrieben |

Der einzige wirklich schwierige Schritt war Schritt 3-4: der Bewertungsalgorithmus. Alles andere war handwerkliche Arbeit. Das ist typisch fuer Software-Projekte: es gibt meistens **einen** zentralen Algorithmus der die eigentliche Denkarbeit erfordert -- der Rest ist Struktur und Verkabelung.

---

**Dateien des Projekts:**

```
logik.py         Spiellogik -- kein I/O, kein GUI
konsole.py       Konsolenversion
gui.py           tkinter-GUI
test_logik.py    pytest-Tests
```

Starten: `python gui.py`